# 🔴 Solution: MLP Backpropagation (NumPy)

In [ ]:
import numpy as np

In [ ]:
# ✅ SOLUTION

def mlp_loss_and_grads(X, labels, params):
    # X: (N, d_in);  labels: (N,) int;  params: list of (W, b)
    N = X.shape[0]
    L = len(params)
    rows = np.arange(N)

    # ---------- forward (cache A_prev and Z for every layer) ----------
    A = X
    cache = []
    for i, (W, b) in enumerate(params):
        Z = A @ W + b
        cache.append((A, Z))
        A = np.maximum(Z, 0) if i < L - 1 else Z
    logits = A

    # ---------- softmax cross-entropy ----------
    z = logits - logits.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    p = exp_z / exp_z.sum(axis=1, keepdims=True)
    loss = float(-np.log(p[rows, labels]).mean())

    # Fused seed gradient: (predicted - actual) / N
    dZ = p.copy()
    dZ[rows, labels] -= 1.0
    dZ /= N

    # ---------- backward ----------
    grads = [None] * L
    for i in range(L - 1, -1, -1):
        A_prev, _ = cache[i]
        W, _ = params[i]

        grads[i] = (A_prev.T @ dZ, dZ.sum(axis=0))   # product rule + broadcast rule

        if i > 0:
            dA_prev = dZ @ W.T
            Z_prev = cache[i - 1][1]
            dZ = dA_prev * (Z_prev > 0)              # ReLU gate

    return loss, grads

In [ ]:
# Verify — gradient check against finite differences, then train
np.random.seed(3)
X = np.random.randn(5, 3)
labels = np.array([0, 1, 2, 1, 0])
params = [(np.random.randn(3, 4), np.random.randn(4)), (np.random.randn(4, 3), np.random.randn(3))]

loss, grads = mlp_loss_and_grads(X, labels, params)
W = params[0][0]
eps = 1e-6
num = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        orig = W[i, j]
        W[i, j] = orig + eps; lp, _ = mlp_loss_and_grads(X, labels, params)
        W[i, j] = orig - eps; lm, _ = mlp_loss_and_grads(X, labels, params)
        W[i, j] = orig
        num[i, j] = (lp - lm) / (2 * eps)

print("max |analytic - numerical|:", np.abs(grads[0][0] - num).max())

In [ ]:
from torch_judge import check
check("numpy_mlp_backward")